# openEO process-graph layers in JupyterGIS

This notebook connects to a local [TiTiler OpenEO](https://github.com/sentinel-hub/titiler-openeo) / OpenEO backend,
builds a small process graph with the [`openeo`](https://cran.r-project.org/package=openeo) R client,
and adds it to a `GISDocument` with `add_openeo_tile_layer()`.

It requires `jupytergis >= 0.16.0a2` (which ships the `OpenEOTileLayer` frontend) and the `openeo` R client.
Both are installed in the `conda`/`pixi` environment.

## Prerequisites

openEO always needs a running backend — there is no public server assumed here, so you run one locally:

1. **A local titiler-openeo server (always required).** Start it with `pixi run serve-openeo`, which launches
   the service on `http://localhost:8080`. See the
   [TiTiler OpenEO local-setup guide](https://sentinel-hub.github.io/titiler-openeo/local-setup/) for how to
   install and configure it (including the `services/*.json` store that lists the collections it exposes).

2. **A Copernicus Data Space account (depends on the server config).** titiler-openeo reads Sentinel data
   straight from Copernicus' S3, so if your configuration points at Copernicus collections you need S3 keys.
   Create an account and generate the S3 credentials at the
   [Copernicus S3 keys manager](https://eodata-s3keysmanager.dataspace.copernicus.eu/panel/s3-credentials),
   then set them in the titiler-openeo environment as described in the local-setup guide. This step is only
   needed when the backend actually serves Copernicus-hosted data.

## Connect and log in

The local dev backend listens on `http://localhost:8080` with basic auth `test` / `test`.

In [ ]:
con <- openeo::connect(host = "http://localhost:8080")
openeo::login(user = "test", password = "test", con = con)

In [ ]:
# Inspect what the backend offers, then pick a collection id below.
collections <- openeo::list_collections(con)
head(collections)

## Build a process graph

Adjust `collection_id`, `bands`, and the spatial/temporal extents to match a collection your backend exposes.

In [ ]:
p <- openeo::processes()

# A globally-covered CLMS collection renders anywhere on land, unlike the
# Copernicus Contributing Missions (ccm-*), whose tasked imagery has sparse
# coverage. fapar_fapar is the Fraction of Absorbed Photosynthetically Active
# Radiation. (Backend also lists "ccm-optical", "ccm-sar", etc.)
collection_id <- "MAXAR_cyclone_emnati22"

# Read the extents straight from the collection metadata (fetched above)
# instead of hardcoding them, so the request always matches what the backend
# holds. `list_collections()` flattens the STAC bbox to
# c(west, south, east, north), and temporal is a list of (start, end) intervals.
collection <- collections[[collection_id]]
bbox <- collection$extent$spatial
interval <- collection$extent$temporal[[1]]

cube <- p$load_collection(
    id = collection_id,
    spatial_extent = list(
        west = bbox[[1]], south = bbox[[2]], east = bbox[[3]], north = bbox[[4]]
    ),
    temporal_extent = c(interval[[1]], interval[[2]]),
    bands = c("fapar_fapar")
)

# Collapse the temporal dimension to a single image. The titiler-openeo tile
# renderer only turns a single-timestep result into a PNG; a multi-date stack
# makes save_result fail with "Expected ImageData object, got RasterStack".
# Reducing over "t" (here: the first observation) yields one renderable image.
image <- p$reduce_dimension(
    data = cube,
    dimension = "t",
    reducer = function(data, context) p$first(data)
)

result <- p$save_result(data = image, format = "PNG")

## Add the openEO layer to a JupyterGIS document

`add_openeo_tile_layer()` serializes the process graph and forwards the backend url and session bearer token, so the frontend's titiler-openeo service can render the tiles.

In [ ]:
doc <- jupytergis::GISDocument$new()
doc

In [ ]:
openeo_layer <- doc$add_openeo_tile_layer(
    result,
    connection = con,
    name = "openEO FAPAR",
    opacity = 1
)